# 🚀 University-1652: LPN (Local Partition Network) Model Test Pipeline

Bu notebook, University-1652 veri seti üzerinde eklediğimiz **LPN (Local Partition Network)** katmanını test etmek üzere hazırlanmıştır.
LPN katmanı, Global Average Pooling yerine görüntüyü bölgesel parçalara bölerek her parçadan bağımsız özellikler çıkarır ve bu sayede modelin bölgesel detayları (bina köşeleri, çatılar vs.) daha iyi öğrenmesini sağlar.

## 1. Ortam Kurulumu ve Veri Seti

In [ ]:
!pip install timm einops scipy

import torch
import torchvision
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Lütfen veri setini Colab'a yükleyin. Google Drive'a kopyaladıysanız aşağıdaki hücreyi kullanarak drive'ı bağlayın.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ÖRNEK: Eğer proje klasörünüz drive'da ise çalışma alanını oraya ayarlayın
import os
os.chdir('/content/drive/MyDrive/University1652-Baseline/cross-view-geo-localization')
print("Aktif dizin:", os.getcwd())
# !ls -l

## 2. LPN Modeli Eğitimi

Eğitim sırasında yeni eklenen parametreleri kullanacağız:
- `--pool lpn`: Global Pooling yerine LPN katmanlarını aktif eder.
- `--lpn_blocks 4`: Görüntünün kaça bölüneceğini belirler (4 parça).
- `--lpn_mode square`: Kare şeklindeki (2x2) veya `horizontal` (1x4 yatay) bölme stratejisi.
- `--batchsize 8`: LPN feature boyutunu büyüttüğü için (2048 x 4 = 8192) CUDA memory taşmalarını engellemek adına batch size düşürülmüştür.

In [ ]:
# LPN Modeli ile Eğitim (Square Mode -> 2x2 blocks)
!python train.py \
    --name lpn_square_test \
    --data_dir ./University-1652 \
    --views 2 \
    --pool lpn \
    --lpn_blocks 4 \
    --lpn_mode square \
    --batchsize 8 \
    --lr 0.01 \
    --droprate 0.5 \
    --fp16

## 3. Test ve Değerlendirme (Inference)

Eğittiğimiz LPN modelinin Retrieval performansını test ediyoruz.

In [ ]:
# Modeli test setinde koşturup özellik vektörlerini çıkartma
!python test.py \
    --name lpn_square_test \
    --test_dir ./University-1652/test \
    --gpu_ids 0 \
    --which_epoch last

In [ ]:
# Test özelliklerinden yola çıkarak mAP, Recall@1 vb metrikleri hesaplama
import scipy.io
import os

print("Rank Performansı Değerlendiriliyor...")
!python evaluate_gpu.py

## 4. (Alternatif) Horizontal LPN Eğitimi

Yukarıdaki model kare olarak bölüyordu (2x2). Alternatif olarak bina cephelerini yatay şeritler halinde öğrenen `horizontal` LPN kurabiliriz.

In [ ]:
# Horizontal LPN Edge Case
# !python train.py \
#     --name lpn_horizontal_test \
#     --data_dir ./University-1652 \
#     --views 2 \
#     --pool lpn \
#     --lpn_blocks 4 \
#     --lpn_mode horizontal \
#     --batchsize 8 \
#     --lr 0.01 \
#     --fp16